# Phase 4: CodeT5+ 220M with LoRA

**Project:** Automated Code Documentation and Bug Report Analysis Using Fine-Tuned CodeT5+ Models

This notebook fine-tunes `Salesforce/codet5p-220m` with LoRA adapters on the same fixed data (`splits_v2`) used for the Phase 3 full fine-tune. Everything is matched to Phase 3 except the learning rate and which parameters train, so the comparison isolates the effect of LoRA.

**Run order:** run every cell from top to bottom in one session. The setup cell mounts Drive and installs `peft`.

**Sections**
1. Setup
2. Config
3. Tokenizer helper
4. Data loading and tokenization
5. LoRA model and efficiency logging
6. Training (resumable)
7. Save adapter and loss curves
8. Generation (adapter latency, then merged model)
9. Evaluation (full docstring and first sentence)
10. Save results
11. Comparison against the Phase 3 full fine-tune


## 1. Setup

In [ ]:
!pip install -q transformers datasets accelerate evaluate sacrebleu rouge_score nltk sentencepiece peft

from google.colab import drive
drive.mount('/content/drive')

import nltk
for pkg in ['wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    nltk.download(pkg, quiet=True)

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (switch runtime to a GPU)')


## 2. Config

Matched to Phase 3: data, sequence lengths, epochs, effective batch size, warmup, weight decay, seed, bf16, and generation settings. Deliberate differences: the LoRA adapters, and the learning rate (LoRA needs a higher one because only a small set of weights trains). State both in the report.

In [ ]:
import os, json, random, time, math, glob, re
import numpy as np
import pandas as pd
import torch

# ---------- Paths ----------
BASE_DIR      = '/content/drive/MyDrive/capstone'
DATA_DIR      = f'{BASE_DIR}/data'
SPLIT_DIR     = f'{DATA_DIR}/splits_v2'
OUTPUT_DIR    = f'{BASE_DIR}/runs/codet5p_220m_lora'
RESULTS_DIR   = f'{BASE_DIR}/results/phase4'
PHASE3_DIR    = f'{BASE_DIR}/results/phase3_v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ---------- Dataset schema ----------
SOURCE_COL = 'input_text'
TARGET_COL = 'target_text'

# ---------- Model ----------
MODEL_NAME     = 'Salesforce/codet5p-220m'
MAX_SOURCE_LEN = 512
MAX_TARGET_LEN = 128

# ---------- LoRA (from the Weeks 1-4 notebook) ----------
LORA_R        = 8
LORA_ALPHA    = 32
LORA_DROPOUT  = 0.1
LORA_TARGETS  = ['q', 'v']     # attention query and value projections

# ---------- Training (matched to Phase 3 except learning rate) ----------
SEED                 = 42
LEARNING_RATE        = 3e-4     # Phase 3 used 5e-5; LoRA typically needs roughly 3e-4 to 1e-3
NUM_EPOCHS           = 3
PER_DEVICE_TRAIN_BS  = 16
PER_DEVICE_EVAL_BS   = 32
GRAD_ACCUM_STEPS     = 2        # effective batch = 32, same as Phase 3
WARMUP_RATIO         = 0.05
WEIGHT_DECAY         = 0.01
LR_SCHEDULER         = 'linear'
USE_BF16             = True

# ---------- Generation (identical to Phase 3) ----------
NUM_BEAMS       = 4
MAX_NEW_TOKENS  = 128
GEN_BATCH_SIZE  = 32

DEBUG_SUBSET = None   # e.g. 2000 for a quick smoke test

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(SEED)


## 3. Tokenizer helper

In [ ]:
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download

def load_tokenizer_safe(model_name, cache_root='/content/model_cache'):
    # Same fix as the Weeks 1-4 notebook: drop added_tokens.json and normalize special_tokens_map.json
    local_dir = os.path.join(cache_root, model_name.replace('/', '__'))
    snapshot_download(repo_id=model_name, local_dir=local_dir)

    added_tokens_path = os.path.join(local_dir, 'added_tokens.json')
    if os.path.exists(added_tokens_path):
        os.remove(added_tokens_path)

    special_tokens_path = os.path.join(local_dir, 'special_tokens_map.json')
    if os.path.exists(special_tokens_path):
        with open(special_tokens_path, 'r', encoding='utf-8') as f:
            special_tokens = json.load(f)
        additional = special_tokens.get('additional_special_tokens')
        if additional and isinstance(additional[0], dict):
            special_tokens['additional_special_tokens'] = [
                t['content'] if isinstance(t, dict) else t for t in additional
            ]
            with open(special_tokens_path, 'w', encoding='utf-8') as f:
                json.dump(special_tokens, f)

    return AutoTokenizer.from_pretrained(local_dir)

tokenizer = load_tokenizer_safe(MODEL_NAME)
print('Tokenizer:', type(tokenizer).__name__, '| vocab size:', len(tokenizer))


## 4. Data loading and tokenization

In [ ]:
from datasets import Dataset

def load_split(name):
    return Dataset.from_pandas(pd.read_parquet(f'{SPLIT_DIR}/{name}.parquet'), preserve_index=False)

train_raw, val_raw, test_raw = load_split('train'), load_split('val'), load_split('test')

if DEBUG_SUBSET:
    train_raw = train_raw.select(range(min(DEBUG_SUBSET, len(train_raw))))
    val_raw   = val_raw.select(range(min(DEBUG_SUBSET, len(val_raw))))
    test_raw  = test_raw.select(range(min(DEBUG_SUBSET, len(test_raw))))
    print('DEBUG subset in use')

print(f'Train: {len(train_raw):,} | Val: {len(val_raw):,} | Test: {len(test_raw):,}')
print(test_raw.to_pandas().groupby(['source', 'language']).size())


In [ ]:
def preprocess(batch):
    model_inputs = tokenizer(batch[SOURCE_COL], max_length=MAX_SOURCE_LEN, truncation=True)
    labels = tokenizer(text_target=batch[TARGET_COL], max_length=MAX_TARGET_LEN, truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tok_train = train_raw.map(preprocess, batched=True, remove_columns=train_raw.column_names, desc='Tokenizing train')
tok_val   = val_raw.map(preprocess,   batched=True, remove_columns=val_raw.column_names,   desc='Tokenizing validation')
print('Tokenized.')


## 5. LoRA model and efficiency logging

In [ ]:
from transformers import AutoModelForSeq2SeqLM, TrainerCallback
from peft import LoraConfig, get_peft_model, TaskType

base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
print('Weights dtype:', next(base_model.parameters()).dtype)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGETS,
    bias='none',
)
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total (incl. adapters): {total_params:,} | Trainable: {trainable_params:,} ({trainable_params/total_params:.3%})')

class EfficiencyCallback(TrainerCallback):
    def __init__(self):
        self.epoch_times, self.epoch_peak_mem_gb = [], []
        self._t0 = None
        self.train_start = None
    def on_train_begin(self, args, state, control, **kw):
        self.train_start = time.time()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
    def on_epoch_begin(self, args, state, control, **kw):
        self._t0 = time.time()
    def on_epoch_end(self, args, state, control, **kw):
        self.epoch_times.append(time.time() - self._t0)
        if torch.cuda.is_available():
            self.epoch_peak_mem_gb.append(torch.cuda.max_memory_allocated() / 1e9)

eff_cb = EfficiencyCallback()


## 6. Training (resumable)

Same recipe as Phase 3: validation loss each epoch, best checkpoint by `eval_loss`, checkpoints on Drive, automatic resume if the session drops.

In [ ]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from transformers.trainer_utils import get_last_checkpoint

collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100, pad_to_multiple_of=8)

steps_per_epoch = math.ceil(len(tok_train) / (PER_DEVICE_TRAIN_BS * GRAD_ACCUM_STEPS))
WARMUP_STEPS = int(WARMUP_RATIO * steps_per_epoch * NUM_EPOCHS)
print(f'Steps/epoch: {steps_per_epoch:,} | total: {steps_per_epoch * NUM_EPOCHS:,} | warmup steps: {WARMUP_STEPS:,}')

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type=LR_SCHEDULER,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BS,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BS,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    bf16=USE_BF16 and torch.cuda.is_available(),
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    greater_is_better=False,
    logging_strategy='steps',
    logging_steps=100,
    predict_with_generate=False,
    label_names=['labels'],           # needed so eval loss works with a PEFT-wrapped model
    dataloader_num_workers=2,
    report_to='none',
    seed=SEED,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=collator,
    processing_class=tokenizer,
    callbacks=[eff_cb],
)

last_ckpt = get_last_checkpoint(OUTPUT_DIR) if os.path.isdir(OUTPUT_DIR) else None
print('Resuming from:', last_ckpt if last_ckpt else 'scratch')

train_result = trainer.train(resume_from_checkpoint=last_ckpt)
print(train_result.metrics)


## 7. Save adapter and loss curves

In [ ]:
BEST_DIR = f'{OUTPUT_DIR}/best_adapter'
trainer.save_model(BEST_DIR)          # saves the LoRA adapter only
tokenizer.save_pretrained(BEST_DIR)

adapter_mb = sum(os.path.getsize(os.path.join(BEST_DIR, f)) for f in os.listdir(BEST_DIR)
                 if f.startswith('adapter_model')) / 1e6
print('Saved adapter to', BEST_DIR, f'| adapter size: {adapter_mb:.1f} MB')
print('Best checkpoint:', trainer.state.best_model_checkpoint)
print('Best eval_loss:', trainer.state.best_metric)


In [ ]:
import matplotlib.pyplot as plt

log = trainer.state.log_history
train_steps  = [l['step'] for l in log if 'loss' in l and 'eval_loss' not in l]
train_losses = [l['loss'] for l in log if 'loss' in l and 'eval_loss' not in l]
eval_steps   = [l['step'] for l in log if 'eval_loss' in l]
eval_losses  = [l['eval_loss'] for l in log if 'eval_loss' in l]

plt.figure(figsize=(8, 4.5))
plt.plot(train_steps, train_losses, label='Train loss', alpha=0.8)
plt.plot(eval_steps, eval_losses, marker='o', label='Validation loss')
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('CodeT5+ 220M LoRA: loss curves')
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/loss_curves_lora.png', dpi=200)
plt.show()


## 8. Generation

Latency is measured twice: with the adapters attached (1,000 test examples), then after merging the adapters into the base weights (full test set). A merged LoRA model has the same architecture and speed as the full fine-tune, and produces the same outputs as the unmerged one.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = trainer.model.to(device).eval()

@torch.no_grad()
def generate_all(texts):
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    outputs = [None] * len(texts)
    for start in range(0, len(order), GEN_BATCH_SIZE):
        idx = order[start:start + GEN_BATCH_SIZE]
        batch = tokenizer([texts[i] for i in idx], max_length=MAX_SOURCE_LEN, truncation=True,
                          padding=True, return_tensors='pt').to(device)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16, enabled=(device == 'cuda' and USE_BF16)):
            gen = model.generate(**batch, num_beams=NUM_BEAMS, max_new_tokens=MAX_NEW_TOKENS, early_stopping=True)
        decoded = tokenizer.batch_decode(gen, skip_special_tokens=True)
        for i, d in zip(idx, decoded):
            outputs[i] = d.strip()
    return outputs

test_sources = test_raw[SOURCE_COL]
test_refs    = test_raw[TARGET_COL]

# 1) Latency with adapters attached (subset)
sub = test_sources[:1000]
t0 = time.time(); _ = generate_all(sub)
lat_unmerged_ms = 1000 * (time.time() - t0) / len(sub)
print(f'Adapters attached: {lat_unmerged_ms:.1f} ms/example (n={len(sub)})')

# 2) Merge adapters into the base weights, then generate the full test set
model = model.merge_and_unload().eval()
t0 = time.time()
test_preds = generate_all(test_sources)
gen_seconds = time.time() - t0
lat_merged_ms = 1000 * gen_seconds / len(test_preds)
print(f'Merged model: generated {len(test_preds):,} predictions in {gen_seconds/60:.1f} min ({lat_merged_ms:.1f} ms/example)')


## 9. Evaluation

Each metric is reported two ways: on the **full docstring** (as stored) and on the **first sentence only**, which is the usual convention for code summarization because the rest of a docstring is mostly parameter and returns boilerplate. The same function scores the Phase 3 predictions in Section 11, so the comparison is consistent.

In [ ]:
import evaluate

bleu_m   = evaluate.load('bleu')
rouge_m  = evaluate.load('rouge')
meteor_m = evaluate.load('meteor')

def first_sentence(text):
    t = text.strip().split('\n\n')[0]
    t = ' '.join(t.split())
    m = re.search(r'(?<=[.!?])\s', t)
    return (t[:m.start()] if m else t).strip()

def score(preds, refs):
    preds = [p if p.strip() else ' ' for p in preds]
    refs  = [r if r.strip() else ' ' for r in refs]
    b = bleu_m.compute(predictions=preds, references=[[r] for r in refs], max_order=4)['bleu']
    r = rouge_m.compute(predictions=preds, references=refs)['rougeL']
    m = meteor_m.compute(predictions=preds, references=refs)['meteor']
    return {'BLEU-4': 100 * b, 'ROUGE-L': 100 * r, 'METEOR': 100 * m}

def score_table(df):
    # df needs columns: reference, prediction, group
    d = df.copy()
    d['reference_fs']  = d['reference'].map(first_sentence)
    d['prediction_fs'] = d['prediction'].map(first_sentence)
    variants = [('full_docstring', 'prediction', 'reference'),
                ('first_sentence', 'prediction_fs', 'reference_fs')]
    rows = []
    def add(label, sub):
        for variant, p, r in variants:
            rows.append({'group': label, 'variant': variant, 'n': len(sub),
                         **score(sub[p].tolist(), sub[r].tolist())})
    add('ALL', d)
    for g, sub in d.groupby('group'):
        add(g, sub)
    return pd.DataFrame(rows).round(2)

df = pd.DataFrame({'source': test_sources, 'reference': test_refs, 'prediction': test_preds})
df['group'] = [('gitbugs' if s == 'gitbugs' else l) for s, l in zip(test_raw['source'], test_raw['language'])]

results = score_table(df)
results


## 10. Save results

In [ ]:
summary = {
    'model': MODEL_NAME,
    'method': 'lora',
    'lora': {'r': LORA_R, 'alpha': LORA_ALPHA, 'dropout': LORA_DROPOUT, 'target_modules': LORA_TARGETS},
    'hyperparameters': {
        'learning_rate': LEARNING_RATE, 'epochs': NUM_EPOCHS,
        'effective_batch_size': PER_DEVICE_TRAIN_BS * GRAD_ACCUM_STEPS,
        'warmup_ratio': WARMUP_RATIO, 'weight_decay': WEIGHT_DECAY,
        'max_source_len': MAX_SOURCE_LEN, 'max_target_len': MAX_TARGET_LEN,
        'bf16': USE_BF16, 'seed': SEED,
    },
    'generation': {'num_beams': NUM_BEAMS, 'max_new_tokens': MAX_NEW_TOKENS},
    'efficiency': {
        'total_params': total_params,
        'trainable_params': trainable_params,
        'trainable_pct': round(100 * trainable_params / total_params, 3),
        'adapter_size_mb': round(adapter_mb, 1),
        'epoch_times_sec': [round(t, 1) for t in eff_cb.epoch_times],
        'total_train_time_min': round((time.time() - eff_cb.train_start) / 60, 1) if eff_cb.train_start else None,
        'peak_gpu_mem_gb': round(max(eff_cb.epoch_peak_mem_gb), 2) if eff_cb.epoch_peak_mem_gb else None,
        'generation_ms_per_example_merged': round(lat_merged_ms, 1),
        'generation_ms_per_example_adapters_attached': round(lat_unmerged_ms, 1),
    },
    'best_eval_loss': trainer.state.best_metric,
    'best_checkpoint': trainer.state.best_model_checkpoint,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
}

results.to_csv(f'{RESULTS_DIR}/metrics_lora.csv', index=False)
df.to_csv(f'{RESULTS_DIR}/predictions_lora.csv', index=False)
with open(f'{RESULTS_DIR}/summary_lora.json', 'w') as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print('\nSaved to', RESULTS_DIR)


## 11. Comparison against the Phase 3 full fine-tune

Loads the saved Phase 3 results from Drive and rescores its predictions with the same function (so the first-sentence numbers are directly comparable). If a file is not found, check `PHASE3_DIR` in the Config cell.

In [ ]:
def find_one(directory, pattern):
    hits = sorted(glob.glob(os.path.join(directory, pattern)))
    assert hits, f'No file matching {pattern} in {directory}'
    return hits[0]

p3_pred_path = find_one(PHASE3_DIR, 'predictions_full_ft*.csv')
p3_sum_path  = find_one(PHASE3_DIR, 'summary_full_ft*.json')
print('Using:', p3_pred_path, '|', p3_sum_path)

p3_df = pd.read_csv(p3_pred_path).fillna('')
p3_results = score_table(p3_df)
p3_summary = json.load(open(p3_sum_path))

full = p3_results.assign(method='full_ft')
lora = results.assign(method='lora')
both = pd.concat([full, lora])

metric_cols = ['BLEU-4', 'ROUGE-L', 'METEOR']
wide = both.pivot_table(index=['variant', 'group'], columns='method', values=metric_cols)
comparison = pd.DataFrame(index=wide.index)
for m in metric_cols:
    comparison[f'{m} full_ft'] = wide[(m, 'full_ft')]
    comparison[f'{m} lora']    = wide[(m, 'lora')]
    comparison[f'{m} diff']    = (wide[(m, 'lora')] - wide[(m, 'full_ft')]).round(2)
comparison.to_csv(f'{RESULTS_DIR}/comparison_full_ft_vs_lora.csv')
comparison


In [ ]:
e3, e4 = p3_summary['efficiency'], summary['efficiency']
eff_table = pd.DataFrame({
    'full_ft': {
        'trainable params': e3['trainable_params'],
        'trainable %': e3['trainable_pct'],
        'train time (min)': e3['total_train_time_min'],
        'avg epoch time (min)': round(np.mean(e3['epoch_times_sec']) / 60, 1),
        'peak GPU memory (GB)': e3['peak_gpu_mem_gb'],
        'latency ms/example': e3['generation_ms_per_example'],
        'best eval loss': round(p3_summary['best_eval_loss'], 4),
    },
    'lora': {
        'trainable params': e4['trainable_params'],
        'trainable %': e4['trainable_pct'],
        'train time (min)': e4['total_train_time_min'],
        'avg epoch time (min)': round(np.mean(e4['epoch_times_sec']) / 60, 1),
        'peak GPU memory (GB)': e4['peak_gpu_mem_gb'],
        'latency ms/example': e4['generation_ms_per_example_merged'],
        'best eval loss': round(summary['best_eval_loss'], 4),
    },
})
eff_table['lora vs full_ft'] = [
    f"{(b / a - 1) * 100:+.1f}%" if isinstance(a, (int, float)) and a else ''
    for a, b in zip(eff_table['full_ft'], eff_table['lora'])
]
eff_table.to_csv(f'{RESULTS_DIR}/efficiency_comparison.csv')
print(f"Adapter size on disk: {e4['adapter_size_mb']} MB")
print('Note: efficiency rows are only comparable if both runs used the same GPU.')
eff_table


### Notes for Phase 5

- `comparison_full_ft_vs_lora.csv` and `efficiency_comparison.csv` feed the results section and the three-model table (add CodeT5-small, retrained on `splits_v2`, using the same scoring function).
- Report both the full-docstring and first-sentence numbers. The full-docstring scores are low for every model because references include parameter and returns sections the models rarely reproduce.
- If LoRA underperforms at the default learning rate, try 5e-4 or raise `LORA_R` to 16, and report the sweep. Keep every other setting fixed.
- State the deliberate differences from Phase 3 (learning rate, trainable parameters) in the methods section.
